In [ ]:
import numpy as np
import pandas as pd
import osmnx as ox

import imn_generation
import imn_loading
from map_matching import map_match_valhalla_all, points_to_osmnx_routes
from personalized_routing import add_edge_attributes

# Loading the data

In [ ]:
# Using a sample dataset of Milan, Italy
points_df = pd.read_csv('data/sample_Milan_2007.csv.gz').rename(columns={
    'userid':'id', 
    'datetime':'timestamp',
    'lon':'longitude', 
    'lat':'latitude'
})

points_df = points_df[points_df['quality'] == 3]

# Convert timestamp string to Epoch in seconds
points_df['timestamp'] = pd.to_datetime(points_df['timestamp']).values.astype(np.int64) // 10 ** 9

In [ ]:
# Download and save the OSM graph for the area of interest
bufsize = 0.01
bbox = (np.min(points_df['longitude'] - bufsize),
         np.min(points_df['latitude'] - bufsize),
         np.max(points_df['longitude'] + bufsize),
         np.max(points_df['latitude'] + bufsize))

G = ox.graph_from_bbox(bbox, network_type='drive', simplify=False)
G = ox.truncate.largest_component(G, strongly=True)
G = add_edge_attributes(G)
ox.save_graphml(G, 'data/milan0_2007_graph.graphml')

# Constructing the IMN + map-matching

In [ ]:
# IMN generation takes care of segmentation
# Also removes blatant errors in trajectories
# and excludes users with too little history (< 10 trajs)
imn_generation.main_from_code(points_df, 'data/milano_2007_imns.json.gz')

In [ ]:
imns = imn_loading.read_imn('data/milano_2007_imns.json.gz')
segmented_trajs = imn_loading.get_trajectories_from_imns(imns)

In [ ]:
# 1st step: map-match
# Must have constructed the tile extracts for Valhalla in advance, see README
map_match_valhalla_all(segmented_trajs,
                       output_path='data/milano_2007_mapmatched.csv.gz',
                       tile_extract='valhalla/valhalla_tiles',
                       verbose=True)

In [ ]:
# 2nd step: convert map-matched points to OSMnx routes
points_to_osmnx_routes(G,
                       input_path='data/milano_2007_mapmatched.csv.gz',
                       output_path='data/milano_2007_routes.csv.gz',
                       verbose=True)

# Collecting route/edge metrics

# Running the iterative method